In [ ]:
from riotwatcher import LolWatcher, ApiError, RiotWatcher
from datetime import datetime
import pandas as pd
import json

In [12]:
lol_watcher = LolWatcher('RGAPI-beff131a-18f7-4f64-bae0-adbc84f66b7e')
riot_watcher = RiotWatcher('RGAPI-beff131a-18f7-4f64-bae0-adbc84f66b7e')
region = 'EUW1'

yo = riot_watcher.account.by_riot_id('EUROPE', 'xAllow', 'ESP')

puuid = yo['puuid']
print(puuid)

TjBX4kwEtJzTOhf-ZdB42jgc8EopzuxdfdFC7hio3dylZRLVc9AiP9H1xRF2j5uezXGvFqe1-mWmKw


### Obtener todas las partidas entre dos fechas usando paginación

El endpoint de Riot limita la cantidad de partidas devueltas por petición (máximo 100). Para obtener todas las partidas en un rango de fechas, es necesario paginar usando el parámetro `start`. Aquí tienes un ejemplo de cómo hacerlo:

In [ ]:
import os

match_region = 'europe'
archivo = 'todas_partidas.json'
inicio_season = '2026-01-07T00:00:00Z'

# Cargar partidas existentes si hay archivo
if os.path.exists(archivo):
    with open(archivo, 'r', encoding='utf-8') as f:
        partidas_guardadas = json.load(f)
    ids_existentes = {p['metadata']['matchId'] for p in partidas_guardadas}
    # Buscar la fecha más reciente guardada
    ultima_fecha = max(p['info']['gameCreation'] for p in partidas_guardadas)
    # Si gameCreation ya es string, convertir; si es int (ms), convertir
    if isinstance(ultima_fecha, str):
        start_epoch = int(datetime.strptime(ultima_fecha, '%Y-%m-%d %H:%M:%S').timestamp())
    else:
        start_epoch = int(ultima_fecha / 1000)
    print(f'Partidas guardadas: {len(partidas_guardadas)}. Descargando desde: {datetime.fromtimestamp(start_epoch)}')
else:
    partidas_guardadas = []
    ids_existentes = set()
    start_epoch = int(datetime.strptime(inicio_season, '%Y-%m-%dT%H:%M:%SZ').timestamp())
    print(f'No hay archivo previo. Descargando desde inicio de season: {inicio_season}')

end_epoch = int(datetime.now().timestamp())

# Obtener IDs de partidas de SoloQ desde la última fecha hasta ahora
nuevas_ids = []
start = 0
count = 100
while True:
    partidas = lol_watcher.match.matchlist_by_puuid(
        match_region, puuid, queue=420, type='ranked',
        start_time=start_epoch, end_time=end_epoch,
        start=start, count=count
    )
    if not partidas:
        break
    nuevas_ids.extend(partidas)
    if len(partidas) < count:
        break
    start += count

# Filtrar IDs que ya tenemos
nuevas_ids = [mid for mid in nuevas_ids if mid not in ids_existentes]
print(f'Nuevas partidas por descargar: {len(nuevas_ids)}')

Partidas guardadas: 392. Descargando desde: 2026-03-16 16:23:35
Nuevas partidas por descargar: 0


In [ ]:
# Descargar detalles de las nuevas partidas y normalizar timestamps/duraciones

def normalizar_duracion_a_segundos(valor):
    if not isinstance(valor, (int, float)):
        return valor
    segundos = valor / 1000
    return int(segundos) if segundos.is_integer() else round(segundos, 3)

nuevas_partidas = []
for i, match_id in enumerate(nuevas_ids):
    match_data = lol_watcher.match.by_id(match_region, match_id)
    info = match_data['info']

    # Según Riot:
    # - si existe gameEndTimestamp, gameDuration ya viene en segundos
    # - si no existe, gameDuration viene en milisegundos y hay que pasarlo a segundos
    if 'gameDuration' in info and isinstance(info['gameDuration'], (int, float)):
        if 'gameEndTimestamp' not in info or info['gameEndTimestamp'] is None:
            info['gameDuration'] = normalizar_duracion_a_segundos(info['gameDuration'])

    # Convertir timestamps absolutos (epoch en ms) a fecha legible
    for campo in ('gameCreation', 'gameStartTimestamp', 'gameEndTimestamp'):
        if campo in info and isinstance(info[campo], (int, float)):
            info[campo] = datetime.fromtimestamp(info[campo] / 1000).strftime('%Y-%m-%d %H:%M:%S')

    nuevas_partidas.append(match_data)
    if (i + 1) % 50 == 0:
        print(f'  Descargadas {i + 1}/{len(nuevas_ids)}...')

# Combinar con las existentes
todas_partidas = partidas_guardadas + nuevas_partidas
print(f'Total partidas: {len(todas_partidas)} ({len(nuevas_partidas)} nuevas)')

Total partidas: 392 (0 nuevas)


In [ ]:
with open('todas_partidas.json', 'w', encoding='utf-8') as f:
    json.dump(todas_partidas, f, indent=4, ensure_ascii=False)

print(f'Guardadas {len(todas_partidas)} partidas en todas_partidas.json')

Guardadas 392 partidas en todas_partidas.json
